# 06 — Cost stack

**Code under test:** `fleximorpv2/models/economics.py`

**Purpose:** Reconcile CAPEX/OPEX components and flag placeholder revenue logic.

Run cells **top to bottom**. Each markdown cell explains the **next code cell** — what it does, what to inspect in the output, and what counts as a pass.

Track overall audit progress in Obsidian (`FlexiMORP Calculation Audit.md`). These notebooks are the lab workbook, not the checklist.

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("error", category=RuntimeWarning)


def _repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "fleximorpv2").is_dir():
            return candidate
    raise RuntimeError(
        "Could not find fleximorp-project root. "
        "Open Jupyter from the repo or a notebooks/ subdirectory."
    )


_repo = _repo_root()
_audit = _repo / "notebooks" / "audit"
for path in (_repo, _audit):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from _audit_helpers import (
    REPO_ROOT,
    OUTPUT_DIR,
    SITE_OUTPUT_DIR,
    reload_fleximorp,
    assert_close,
    assert_energy_balance,
    assert_cf_bounds,
    assert_positive,
)

reload_fleximorp()
np.random.seed(42)
print(f"Repo root: {REPO_ROOT}")
print(f"Audit outputs: {OUTPUT_DIR}")
print(f"Site outputs: {SITE_OUTPUT_DIR}")

## Step 1 — Cost breakdown sums

**Run the next cell** with a fixed design + performance dict.

**Pass if:**
- `total_capex` equals sum of technology + platform + installation + grid + development lines
- Grid cost grows when `distance_to_shore` increases
- Note: `generation_profile=np.ones(8760)` is a placeholder — revenue timing is not hourly-realistic

In [ ]:
from fleximorpv2.config import load_config
from fleximorpv2.models.economics import EconomicModel

config = load_config("eastport")
economics = EconomicModel(config)
design_vars = {"platform_area": 10_000, "water_depth": 30, "distance_to_shore": 2}
tech_perf = {
    "total_technology_capex": 2e7,
    "total_technology_opex": 4e5,
    "annual_energy": 50_000,
}
metrics = economics.calculate_economics(design_vars, tech_perf)
print({k: v for k, v in metrics.items() if "capex" in k or "opex" in k or k in ("capex", "opex")})
# TODO: assert component sum == metrics["capex"]